In [14]:
import pandas as pd 
import matplotlib.pyplot as plt 
from datetime import datetime
import seaborn as sns

In [15]:
house_energy_profile = pd.read_csv("household_energy_profiles.csv")
solar_wind_grid = pd.read_csv("solar_wind_grid.csv")

In [16]:
house_energy_profile.head()

,household_id,building_type,heating_type,has_solar_panels,has_electric_vehicle,n_residents,area_sqm,smart_meter_installed,monthly_consumption_kwh,energy_poverty_flag
0,HH_0001,Saniert,Wärmepumpe,1,0,2,77,1,202.5,0
1,HH_0002,Saniert,Ölheizung,0,0,2,77,0,394.5,0
2,HH_0003,Passivhaus,Wärmepumpe,0,0,4,77,1,156.8,0
3,HH_0004,Altbau,Gasheizung,0,1,1,77,1,567.8,0
4,HH_0005,Altbau,Gasheizung,1,0,2,77,1,389.5,0


In [17]:
print(house_energy_profile.shape)
house_energy_profile.info()

(500, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   household_id             500 non-null    object 
 1   building_type            500 non-null    object 
 2   heating_type             500 non-null    object 
 3   has_solar_panels         500 non-null    int64  
 4   has_electric_vehicle     500 non-null    int64  
 5   n_residents              500 non-null    int64  
 6   area_sqm                 500 non-null    int64  
 7   smart_meter_installed    500 non-null    int64  
 8   monthly_consumption_kwh  500 non-null    float64
 9   energy_poverty_flag      500 non-null    int64  
dtypes: float64(1), int64(6), object(3)
memory usage: 39.2+ KB


no missing entries

In [18]:
energy_columns_binary = ["has_solar_panels", "has_electric_vehicle","smart_meter_installed", "energy_poverty_flag"]
energy_columns_numeric = ["n_residents", "area_sqm", "monthly_consumption_kwh"]
energy_columns_object = [c for c in house_energy_profile.columns if house_energy_profile[c].dtype == "object"]

In [19]:
house_energy_profile[energy_columns_numeric].describe().T

,count,mean,std,min,25%,50%,75%,max
n_residents,500.0,2.4760,1.104474,1.0,2.000,2.00,3.000,5.0
area_sqm,500.0,77.0000,0.000000,77.0,77.000,77.00,77.000,77.0
monthly_consumption_kwh,500.0,441.9346,218.074301,50.0,270.875,406.75,570.875,1176.8


In [20]:
print(solar_wind_grid.shape)
solar_wind_grid.head()

(2160, 11)


,timestamp,plant_id,plant_type,power_output_mw,capacity_mw,irradiance_wm2,wind_speed_ms,temperature_c,cloud_cover_pct,grid_demand_mw,co2_saved_kg
0,2025-06-01 00:00:00,SOLAR_MUNICH_01,solar,0.016,5.0,0.0,3.96,25.0,54.8,2.565,13.1
1,2025-06-01 01:00:00,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,6.82,23.6,23.6,2.275,0.0
2,2025-06-01 02:00:00,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,5.78,18.0,55.7,1.792,0.0
3,2025-06-01 03:00:00,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,16.69,18.2,14.8,2.274,0.0
4,2025-06-01 04:00:00,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,6.38,14.8,74.1,2.154,0.0


In [33]:
solar_wind_grid["plant_id"].value_counts()

plant_id
SOLAR_MUNICH_01       720
WIND_AUGSBURG_02      720
SOLAR_NUERNBERG_03    720
Name: count, dtype: int64

drei verschiedene plants


seperate date and hour for tidy data format:

In [21]:
solar_wind_grid[["date", "time"]] = solar_wind_grid["timestamp"].str.split(" ", expand=True)
solar_wind_grid.drop("timestamp", axis=1, inplace=True)

solar_wind_grid["date"] = pd.to_datetime(solar_wind_grid["date"])
solar_wind_grid["day_of_the_week"] = solar_wind_grid["date"].dt.day_name()
solar_wind_grid["day_of_the_month"] = solar_wind_grid["date"].dt.day
solar_wind_grid["month"] = solar_wind_grid["date"].dt.month

In [22]:
solar_columns_numeric = ["power_output_mw", "capacity_mw", "irradiance_wm2", "wind_speed_ms", "temperature_c", "cloud_cover_pct", "grid_demand_mw", "co2_saved_kg"]
solar_columns_object = [c for c in solar_wind_grid.columns if solar_wind_grid[c].dtype == "object"]

In [23]:
solar_wind_grid[solar_columns_numeric].describe().T

,count,mean,std,min,25%,50%,75%,max
power_output_mw,2160.0,1.581556,2.885438,0.00,0.04800,0.9555,2.21025,68.000
capacity_mw,2160.0,5.733333,1.636091,4.20,4.20000,5.0000,8.00000,8.000
irradiance_wm2,2160.0,254.085972,251.176891,0.00,0.00000,209.8500,451.42500,910.300
wind_speed_ms,2160.0,9.195833,3.672282,2.25,6.52000,8.7100,11.47250,22.880
temperature_c,2160.0,20.002130,6.028461,6.30,14.80000,19.9000,25.30000,33.700
cloud_cover_pct,2160.0,40.429861,20.289142,0.80,24.10000,39.0000,54.90000,94.800
grid_demand_mw,2160.0,3.347767,0.903137,1.06,2.57075,3.3205,4.10300,5.572
co2_saved_kg,2160.0,1296.874398,2366.054790,0.00,39.52500,783.6000,1812.52500,55760.000


In [24]:
def calculate_capacity_factor(actual_output, rated_capacity):
    capacity_factor = actual_output/rated_capacity
    return capacity_factor


In [25]:
solar_wind_grid["capacity_factor"] = calculate_capacity_factor(solar_wind_grid["power_output_mw"], solar_wind_grid["capacity_mw"])

In [27]:
solar_wind_grid.head()

,plant_id,plant_type,power_output_mw,capacity_mw,irradiance_wm2,wind_speed_ms,temperature_c,cloud_cover_pct,grid_demand_mw,co2_saved_kg,date,time,day_of_the_week,day_of_the_month,month,capacity_factor
0,SOLAR_MUNICH_01,solar,0.016,5.0,0.0,3.96,25.0,54.8,2.565,13.1,2025-06-01,00:00:00,Sunday,1,6,0.0032
1,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,6.82,23.6,23.6,2.275,0.0,2025-06-01,01:00:00,Sunday,1,6,0.0000
2,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,5.78,18.0,55.7,1.792,0.0,2025-06-01,02:00:00,Sunday,1,6,0.0000
3,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,16.69,18.2,14.8,2.274,0.0,2025-06-01,03:00:00,Sunday,1,6,0.0000
4,SOLAR_MUNICH_01,solar,0.000,5.0,0.0,6.38,14.8,74.1,2.154,0.0,2025-06-01,04:00:00,Sunday,1,6,0.0000


In [36]:
cf_by_hour = solar_wind_grid.groupby(["plant_id", "time"])["capacity_factor"].mean().unstack(level=0)
cf_by_hour            

plant_id,SOLAR_MUNICH_01,SOLAR_NUERNBERG_03,WIND_AUGSBURG_02
time,,,
00:00:00,0.003800,0.004635,0.235737
01:00:00,0.001933,0.010865,0.396850
02:00:00,0.004187,0.005508,0.217517
03:00:00,0.003220,0.005222,0.271413
04:00:00,0.004240,0.004167,0.124054
05:00:00,0.015747,0.027643,0.517671
06:00:00,0.156467,0.125032,0.222679
07:00:00,0.246047,0.234651,0.300142
08:00:00,0.364220,0.326643,0.281679


In [43]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

cf_by_dow = cf_by_hour = solar_wind_grid.groupby(["plant_id", "day_of_the_week"])["capacity_factor"].mean().unstack(level=0).reindex(order)
cf_by_dow

plant_id,SOLAR_MUNICH_01,SOLAR_NUERNBERG_03,WIND_AUGSBURG_02
day_of_the_week,,,
Monday,0.274877,0.271889,0.281199
Tuesday,0.249156,0.273829,0.212124
Wednesday,0.264321,0.257981,0.278320
Thursday,0.295767,0.331714,0.298715
Friday,0.262642,0.246414,0.255936
Saturday,0.258821,0.252855,0.380051
Sunday,0.263943,0.312206,0.256048
